### First set of imports 

In [1]:
import pandas as pd 
import json
import os
import numpy as np
import torch
import re

pd.set_option("display.max_colwidth", None) # Just allows me to see full columns for dataframes



### Convert videos json to df

In [2]:
#Get the videos json
videos = os.path.join("..", "videos_all_languages.json")


#Open the json
with open(videos, "r", encoding="utf-8") as f:
    data = json.load(f)

#Turn the json into a pandas dataframe with proper headings     
df = pd.json_normalize(data["videos"])

print(df.columns)
print(df.head())

### Get all the handlandmark paths 

Index(['id', 'language', 'Sign', 'filepath', 'HamNoSys', 'concept_url',
       'HandLandmark', 'annotated_video'],
      dtype='str')
      id language      Sign  \
0  BSL_1      BSL     April   
1  BSL_2      BSL    Athens   
2  BSL_3      BSL    August   
3  BSL_4      BSL    Berlin   
4  BSL_5      BSL  February   

                                              filepath  \
0     /home/ubuntu/Project/Data/External/BSL/April.mp4   
1    /home/ubuntu/Project/Data/External/BSL/Athens.mp4   
2    /home/ubuntu/Project/Data/External/BSL/August.mp4   
3    /home/ubuntu/Project/Data/External/BSL/Berlin.mp4   
4  /home/ubuntu/Project/Data/External/BSL/February.mp4   

                                          HamNoSys  \
0     
1                 
2         
3                                         
4                             

     

### Get landmark paths

In [3]:
#find the path to each landmark json in the videos json
Hand_Landmarks_paths = df["HandLandmark"]

Hand_Landmarks = [p for p in Hand_Landmarks_paths if os.path.exists(p)]



### For each frame in video get landmarks

In [4]:

REDUCED_FACE_LANDMARKS = (
    list(range(70, 103)) +   # eyebrows
    list(range(0, 17)) +     # outer lips
    list(range(61, 88)) +    # inner lips
    list(range(152, 172))    # jawline
)

HAND_LANDMARK_IDS = list(range(21))  # 21 per hand
POSE_LANDMARK_IDS = list(range(33))  # 33 pose points

def convert_frames_to_sequence(frames):
    rows = []

    for frame in frames:
        frame_features = []

        # ---------------- HANDS ----------------
        # Build dicts for left and right hand
        left = {lm["id"]: lm for hand in frame["hands"] if hand["handedness"] == "Left" for lm in hand["landmarks"]}
        right = {lm["id"]: lm for hand in frame["hands"] if hand["handedness"] == "Right" for lm in hand["landmarks"]}

        # Left hand (21 landmarks)
        for i in HAND_LANDMARK_IDS:
            if i in left:
                lm = left[i]
                frame_features.extend([lm["x"], lm["y"], lm["z"]])
            else:
                frame_features.extend([0.0, 0.0, 0.0])

        # Right hand (21 landmarks)
        for i in HAND_LANDMARK_IDS:
            if i in right:
                lm = right[i]
                frame_features.extend([lm["x"], lm["y"], lm["z"]])
            else:
                frame_features.extend([0.0, 0.0, 0.0])

        # ---------------- POSE ----------------
        pose_dict = {lm["id"]: lm for pose in frame["pose"] for lm in pose["landmarks"]}

        for i in POSE_LANDMARK_IDS:
            if i in pose_dict:
                lm = pose_dict[i]
                frame_features.extend([lm["x"], lm["y"], lm["z"]])
            else:
                frame_features.extend([0.0, 0.0, 0.0])

        # ---------------- FACE (reduced) ----------------
        face_dict = {lm["id"]: lm for face in frame["face"] for lm in face["landmarks"]}

        for i in REDUCED_FACE_LANDMARKS:
            if i in face_dict:
                lm = face_dict[i]
                frame_features.extend([lm["x"], lm["y"], lm["z"]])
            else:
                frame_features.extend([0.0, 0.0, 0.0])

        rows.append(frame_features)

    seq = np.array(rows, dtype=np.float32)

    # Downsample
    #seq = seq[::3]

    return seq


In [5]:
import unicodedata

def fix_mojibake(name):
    # 1. Try the classic UTF-8 → Latin-1 fix
    try:
        name = name.encode("latin1").decode("utf8")
    except:
        pass

    # 2. Manual replacements for common mojibake patterns
    replacements = {
        # German
        "ÃŸ": "ß", "Ã„": "Ä", "Ã–": "Ö", "Ãœ": "Ü",
        "Ã¤": "ä", "Ã¶": "ö", "Ã¼": "ü",

        # French
        "Ã©": "é", "Ã¨": "è", "Ãª": "ê", "Ã«": "ë",
        "Ã´": "ô", "Ã¢": "â", "Ã¹": "ù", "Ã»": "û",
        "Ã§": "ç",

        # Greek (common mojibake)
        "Î±": "α", "Î²": "β", "Î³": "γ", "Î´": "δ",
        "Îµ": "ε", "Î¶": "ζ", "Î·": "η", "Î¸": "θ",
        "Î¹": "ι", "Îº": "κ", "Î»": "λ", "Î¼": "μ",
        "Î½": "ν", "Î¾": "ξ", "Î¿": "ο", "Îπ": "π",
        "Ïƒ": "σ", "Ï„": "τ", "Ï…": "υ", "Ï†": "φ",
        "Ï‡": "χ", "Ïˆ": "ψ", "Ï‰": "ω",
        "Î ": "Π", "Î£": "Σ", "Î¤": "Τ", "Î¨": "Ψ",
        "Î©": "Ω",
    }

    for bad, good in replacements.items():
        name = name.replace(bad, good)

    # 3. Normalise Unicode (fixes combining accents)
    name = unicodedata.normalize("NFC", name)

    return name


In [ ]:
os.makedirs("processed_sequences", exist_ok=True)

for path in Hand_Landmarks:
    with open(path, "r") as f:
        data = json.load(f)

    raw = data["video_path"]

    video_id = fix_mojibake(
        os.path.basename(raw)
        .replace(".mp4", "")
        .replace(".MOV", "")
        .replace("_cropped", "")
    )

    seq = convert_frames_to_sequence(data["frames"])

    np.save(f"processed_sequences/{video_id}.npy", seq)


In [ ]:
df["filepath"] = (
    df["filepath"]
    .apply(lambda p: fix_mojibake(os.path.basename(p).replace(".mp4", "").replace(".MOV", "")))
)



In [ ]:
label_dict = dict(zip(df["filepath"], df["HamNoSys"]))


X = []
y = []

for seq_file in os.listdir("processed_sequences"):
    vid = seq_file.replace(".npy", "")
    clean_vid = fix_mojibake(vid)

    if clean_vid in label_dict:
        X.append(np.load(f"processed_sequences/{seq_file}"))
        y.append(label_dict[clean_vid])
    else:
        print("Missing label for:", vid)


In [ ]:
max_len = max(seq.shape[0] for seq in X)
num_features = X[0].shape[1]

x = np.zeros((len(X), max_len, num_features), dtype=np.float32)

for i, seq in enumerate(X):
    x[i, :seq.shape[0], :] = seq


In [ ]:
X_tensor = torch.tensor(x, dtype=torch.float32)


In [ ]:
all_tokens = sorted({c for seq in y for c in seq})
token_to_id = {t: i+3 for i, t in enumerate(all_tokens)}
token_to_id["<PAD>"] = 0
token_to_id["<SOS>"] = 1
token_to_id["<EOS>"] = 2

vocab_size = len(token_to_id)
id_to_token = {v: k for k, v in token_to_id.items()}



In [ ]:
y_sequences = []

for seq in y:
    encoded = [token_to_id["<SOS>"]] + [token_to_id[c] for c in seq] + [token_to_id["<EOS>"]]
    y_sequences.append(encoded)


In [ ]:
import torch
import torchvision
import torchaudio
import lightning as L

print(torch.__version__)
print(torchvision.__version__)
print(torchaudio.__version__)


2.2.2+cu121
0.17.2+cu121
2.2.2+cu121


In [ ]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import Adam #Do I keep adam or GD
import matplotlib.pyplot as plt
import seaborn as sns
import lightning as L
from torch.utils.data import TensorDataset, DataLoader

import random
from sklearn.model_selection import train_test_split
import math 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

sos_id = token_to_id["<SOS>"]
eos_id = token_to_id["<EOS>"]
pad_id = token_to_id["<PAD>"]

print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")




Using device: cuda
True
1
NVIDIA H100 PCIe


In [ ]:
npy_ids = [seq_file.replace(".npy", "") for seq_file in os.listdir("processed_sequences")]

missing = [vid for vid in npy_ids if fix_mojibake(vid) not in label_dict]

print("Missing labels:", len(missing))
print(missing[:30])


Missing labels: 0
[]


### Splitting Data into Train, Val, Test

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_sequences, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

### Trims rubbish from sequence

### Creates windows so that we can expand dataset

In [ ]:
def temporal_subsample(seq, rates=[2, 3]):
    """
    Given a sequence of shape (T, F), return augmented versions
    by taking every k-th frame.
    """
    augmented = []

    for r in rates:
        # Only keep if the subsampled sequence is long enough
        if seq.shape[0] // r > 5:
            augmented.append(seq[::r])

    return augmented

def random_frame_drop(seq, drop_prob=0.05):
    mask = np.random.rand(seq.shape[0]) > drop_prob
    return seq[mask]


# -----------------------------
# APPLY TEMPORAL SUBSAMPLING + FRAME DROP
# -----------------------------
X_aug = []
y_aug = []

for x_seq, y_seq in zip(X_train, y_train):

    # original full sequence
    X_aug.append(x_seq)
    y_aug.append(y_seq)

    # subsampled versions
    for new_seq in temporal_subsample(x_seq):
        X_aug.append(new_seq)
        y_aug.append(y_seq)

    # random frame drop version
    dropped = random_frame_drop(x_seq)
    if dropped.shape[0] > 5:   # ensure not empty
        X_aug.append(dropped)
        y_aug.append(y_seq)



### Padding sequences 

In [ ]:

def pad_sequences_X(seqs):
    max_len = max(seq.shape[0] for seq in seqs)
    num_features = seqs[0].shape[1]

    X = torch.zeros((len(seqs), max_len, num_features), dtype=torch.float32)

    for i, seq in enumerate(seqs):
        X[i, :seq.shape[0]] = torch.tensor(seq, dtype=torch.float32)

    return X

def pad_sequences_y(labels, pad_token=0):
    max_len = max(len(seq) for seq in labels)
    Y = torch.full((len(labels), max_len), pad_token, dtype=torch.long)

    for i, seq in enumerate(labels):
        Y[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)

    return Y



X_train_tensor = pad_sequences_X(X_aug)
y_train_tensor = pad_sequences_y(y_aug)

X_val_tensor = pad_sequences_X(X_val)
y_val_tensor = pad_sequences_y(y_val)

X_test_tensor = pad_sequences_X(X_test)
y_test_tensor = pad_sequences_y(y_test)



### Wraps Tensors

In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

### Batches of 16 of the wrapped Tensors

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset,batch_size=16)

### Encoder Class - Summary of frames for decoder class

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)


In [ ]:
class EncoderBiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            bidirectional=True,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim * 2, hidden_dim)

    def forward(self, x):
        outputs, _ = self.lstm(x)
        outputs = self.fc(outputs)
        return outputs


In [ ]:
class CustomTransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.2):
        super().__init__()

        self.self_attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True
        )
        self.cross_attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True
        )

        self.linear1 = nn.Linear(d_model, d_model * 4)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_model * 4, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask, tgt_key_padding_mask):
        # Self-attention
        _tgt, _ = self.self_attn(
            tgt, tgt, tgt,
            attn_mask=tgt_mask,
            key_padding_mask=tgt_key_padding_mask
        )
        tgt = self.norm1(tgt + self.dropout1(_tgt))

        # Cross-attention
        _tgt, cross_attn_weights = self.cross_attn(
            tgt, memory, memory,
            key_padding_mask=None   # memory has no padding
        )
        tgt = self.norm2(tgt + self.dropout2(_tgt))

        # Feed-forward
        ff = self.linear2(self.dropout(F.relu(self.linear1(tgt))))
        tgt = self.norm3(tgt + self.dropout3(ff))

        return tgt, cross_attn_weights



### Decoder Class - Looks at the memory and assigns tokens

In [ ]:
class CustomTransformerDecoder(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers=4, num_heads=8, dropout=0.2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.positional_encoding = PositionalEncoding(hidden_dim)

        self.layers = nn.ModuleList([
            CustomTransformerDecoderLayer(hidden_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, tgt, memory, tgt_mask, tgt_key_padding_mask):
        x = self.embedding(tgt)
        x = self.positional_encoding(x)

        cross_attn_all_layers = []

        for layer in self.layers:
            x, cross_attn = layer(
                x, memory,
                tgt_mask,
                tgt_key_padding_mask
            )
            cross_attn_all_layers.append(cross_attn)

        logits = self.fc_out(x)
        return logits, cross_attn_all_layers

### Loss function

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=token_to_id["<PAD>"], label_smoothing = 0.1) #Loss function

### Seq2seq - Runs encoder and decoder

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, input_dim, vocab_size, hidden_dim=256):
        super().__init__()
        self.encoder = EncoderBiLSTM(input_dim, hidden_dim)
        self.decoder = CustomTransformerDecoder(vocab_size, hidden_dim)

    def generate_square_subsequent_mask(self, size):
        mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
        return mask

    def forward(self, src, tgt):
        # Encode
        memory = self.encoder(src)

        # Masks
        tgt_mask = self.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        tgt_key_padding_mask = (tgt == pad_id)

        # Decode
        logits, cross_attn = self.decoder(
            tgt,
            memory,
            tgt_mask,
            tgt_key_padding_mask
        )

        return logits, cross_attn



In [ ]:
def compute_coverage_loss(cross_attn_layers):
    coverage_loss = 0.0

    # Initialize coverage vector
    coverage = None

    for attn in cross_attn_layers:
        attn = attn.mean(dim=1)  

        if coverage is None:
            coverage = attn
        else:
            coverage_loss += torch.sum(torch.min(attn, coverage))
            coverage = coverage + attn

    return coverage_loss


### Connects it all together

In [ ]:
num_features = X_tensor.shape[2]  # detect input features
hidden_size = 256

model = Seq2Seq(
    input_dim=num_features,
    vocab_size=vocab_size,
    hidden_dim=256
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr = 5e-5)



### Updated weights based on loss function

In [ ]:
def token_accuracy(output, trg, pad_idx=0):
    
    pred = output.argmax(1)
    mask = trg != pad_idx
    correct = (pred == trg) & mask
    accuracy = correct.sum().item()/mask.sum().item()
    return accuracy*100

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, epoch):
    model.train()
    total_loss = 0

    for src, tgt in dataloader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()

        # Teacher forcing: one forward pass
        batch_size, tgt_len = tgt.size()
        decoder_input = tgt[:, :1]  # <SOS>

        teacher_forcing_ratio = max(0.2, 1 - epoch * 0.01)


        for t in range(1, tgt_len):
            logits, _ = model(src, decoder_input)
            next_token_logits = logits[:, -1, :]
            next_token = next_token_logits.argmax(dim=-1)

            use_teacher = (torch.rand(batch_size, device=device) < teacher_forcing_ratio)
            next_input = torch.where(use_teacher, tgt[:, t], next_token)

            decoder_input = torch.cat([decoder_input, next_input.unsqueeze(1)], dim=1)

        # final forward pass for loss
        logits, cross_attn = model(src, tgt[:, :-1])


        ce_loss = criterion(
            logits.reshape(-1, vocab_size),
            tgt[:, 1:].reshape(-1)
        )

        cov_loss = compute_coverage_loss(cross_attn)
        cov_loss = cov_loss / (tgt.size(0) * tgt.size(1) + 1e-8)  # normalise per token
        loss = ce_loss + 0.05 * cov_loss

        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [ ]:
# -----------------------------
# TRAINING CONFIG
# -----------------------------
num_epochs = 150        
patience = 50             # early stopping patience
best_val_loss = float('inf')
patience_counter = 0

print("Starting training...")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # -----------------------------
    # TRAIN
    # -----------------------------
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        epoch
    )
    print(f"  Train Loss: {train_loss:.4f}")

    # -----------------------------
    # VALIDATION
    # -----------------------------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)

            # forward pass for validation
            logits, cross_attn = model(src, tgt[:, :-1])

            loss = criterion(
                logits.reshape(-1, vocab_size),
                tgt[:, 1:].reshape(-1)
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)
    print(f"  Val Loss:   {val_loss:.4f}")

    # -----------------------------
    # EARLY STOPPING + CHECKPOINT
    # -----------------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0

        torch.save(model.state_dict(), "best_model.pt")
        print("Improvement — best model saved.")
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{patience})")

        if patience_counter >= patience:
            print("\nEarly stopping triggered — training stopped.")
            break

print("\nTraining complete.")


Starting training...

Epoch 1/150


In [ ]:
def beam_search(model, src, sos_id, eos_id, beam_size=3, max_len=200):
    device = src.device
    memory = model.encoder(src)

    sequences = [[sos_id]]
    scores = torch.zeros(1, device=device)

    for _ in range(max_len):
        all_candidates = []

        if all(seq[-1] == eos_id for seq in sequences):
            break

        for i, seq in enumerate(sequences):
            if seq[-1] == eos_id:
                all_candidates.append((seq, scores[i]))
                continue

            tgt = torch.tensor(seq, device=device).unsqueeze(0)
            tgt_mask = model.generate_square_subsequent_mask(len(seq)).to(device)

            tgt_key_padding_mask = (tgt == pad_id)

            logits, _ = model.decoder(
                tgt,
                memory,
                tgt_mask,
                tgt_key_padding_mask
            )

            probs = torch.log_softmax(logits[:, -1, :], dim=-1)

            topk = torch.topk(probs, beam_size)

            for k in range(beam_size):
                token = topk.indices[0][k].item()
                score = scores[i] + topk.values[0][k]

                new_seq = seq + [token]

                # Length normalization
                norm_score = score / ((5 + len(new_seq)) / 6)

                all_candidates.append((new_seq, norm_score))

        ordered = sorted(all_candidates, key=lambda x: x[1], reverse=True)
        sequences = [seq for seq, score in ordered[:beam_size]]
        scores = torch.tensor([score for seq, score in ordered[:beam_size]], device=device)

    return sequences[0]


In [ ]:
def decode_ids(id_list, id_to_token):
    return [id_to_token[i] for i in id_list]


In [ ]:
preds = []
trues = []

model.eval()
with torch.no_grad():
    for src, tgt in val_loader:
        src = src.to(device)
        tgt = tgt.to(device)

        for i in range(src.size(0)):
            pred_seq = beam_search(model, src[i].unsqueeze(0), sos_id, eos_id)

            if eos_id in pred_seq:
                pred_seq = pred_seq[:pred_seq.index(eos_id)]

            preds.append(pred_seq)

            true_seq = tgt[i].tolist()
            if eos_id in true_seq:
                true_seq = true_seq[:true_seq.index(eos_id)]

            trues.append(true_seq)

decoded_preds = [decode_ids(p, id_to_token) for p in preds]
decoded_trues = [decode_ids(t, id_to_token) for t in trues]



: 

: 

: 

### Validation

In [ ]:
import sys
print(sys.executable)


: 

: 

: 

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np
import Levenshtein as Lev


In [ ]:
smooth = SmoothingFunction().method1

def compute_bleu(pred, true):
    return {
        "BLEU-1": sentence_bleu([true], pred, weights=(1,0,0,0), smoothing_function=smooth),
        "BLEU-2": sentence_bleu([true], pred, weights=(0.5,0.5,0,0), smoothing_function=smooth),
        "BLEU-4": sentence_bleu([true], pred, weights=(0.25,0.25,0.25,0.25), smoothing_function=smooth),
    }


In [ ]:
def compute_cer(pred, true):
    pred_str = "".join(pred)
    true_str = "".join(true)
    return Lev.distance(pred_str, true_str) / len(true_str)


In [ ]:
def evaluate(predictions, ground_truth):
    bleu_scores = []
    cer_scores = []
    exact = 0

    for pred, true in zip(predictions, ground_truth):
        bleu = compute_bleu(pred, true)
        cer = compute_cer(pred, true)

        bleu_scores.append(bleu)
        cer_scores.append(cer)

        if pred == true:
            exact += 1

    avg_bleu = {
        "BLEU-1": np.mean([b["BLEU-1"] for b in bleu_scores]),
        "BLEU-2": np.mean([b["BLEU-2"] for b in bleu_scores]),
        "BLEU-4": np.mean([b["BLEU-4"] for b in bleu_scores]),
    }

    avg_cer = np.mean(cer_scores)
    exact_acc = exact / len(predictions)

    return avg_bleu, avg_cer, exact_acc


In [ ]:
decoded_preds = [decode_ids(p, id_to_token) for p in preds]
decoded_trues = [decode_ids(t, id_to_token) for t in trues]

avg_bleu, avg_cer, exact_acc = evaluate(decoded_preds, decoded_trues)

print("BLEU:", avg_bleu)
print("CER:", avg_cer)
print("Exact Match Accuracy:", exact_acc)


BLEU: {'BLEU-1': 0.2505876707663053, 'BLEU-2': 0.11212399197040723, 'BLEU-4': 0.03852027126387747}
CER: 0.9186975808300558
Exact Match Accuracy: 0.0
